In [ ]:
import pandas as pd
import re

# 1. Load the cleaned OCR results from a Parquet file on GitHub
url = 'https://raw.githubusercontent.com/LCM-S25-3035/Chatter/dev/1.data/processed/nicolas/ocr_results_cleaned.parquet'
df = pd.read_parquet(url, engine='pyarrow')

# 2. Extract group identifier and fragment order from the file name in 'Path'
#    Example Path: "00163fe7688e71ce06f495a6811fef71_1.jpg"
df['group_id'] = df['Path'].str.extract(r'(.+?)_\d+\.jpg')
df['fragment_order'] = df['Path'].str.extract(r'_(\d+)\.jpg').astype(int)

# 3. Define a function to merge all fragments of the same group
def merge_fragments(group):
    # 3.1 Sort the fragments by their order
    group = group.sort_values('fragment_order')
    
    # 3.2 Concatenate the cleaned OCR text in sequence, separated by spaces
    merged_text = ' '.join(group['OCR_Text_Cleaned'])
    
    # 3.3 Sum the character counts after cleaning
    total_length = group['Length_After'].sum()
    
    # 3.4 Keep arbitrary values for Format and Size from the first fragment
    fmt = group.iloc[0]['Format']
    size = group.iloc[0]['Size']
    
    # 3.5 Keep the Label from the first fragment (if present)
    label = group.iloc[0].get('Label', None)
    
    # 3.6 Use the group_id value from the first row (all rows share it)
    gid = group.iloc[0]['group_id']
    
    # 3.7 Return a Series representing the merged record
    return pd.Series({
        'Label': label,
        'group_id': gid,
        'Format': fmt,
        'Size': size,
        'OCR_Text_Merged': merged_text,
        'Length_After': total_length
    })

# 4. Apply the merging function to each group
merged_df = df.groupby('group_id', group_keys=False).apply(merge_fragments).reset_index(drop=True)

# 5. Define a function to clean merged text for embeddings / AI processing
def clean_for_embedding(text: str) -> str:
    # 5.1 Remove everything before the actual article content
    text = re.sub(r'^.*?From Wikipedia, the free encyclopedia', '', text, flags=re.S)
    # 5.2 Remove footer starting with "This page was last edited"
    text = re.sub(r'This page was last edited.*$', '', text, flags=re.S)
    
    # 5.3 Remove in-text reference markers like [1], [edit], {eit}, etc.
    text = re.sub(r'\[\d+\]', '', text)      # numeric references
    text = re.sub(r'\[\w+\]', '', text)      # word-based markers
    text = re.sub(r'\{[^\}]*\}', '', text)   # braces {…}
    
    # 5.4 Lowercase the text for normalization
    text = text.lower()
    
    # 5.5 Remove URLs and email addresses
    text = re.sub(r'http\S+|\S+@\S+', '', text)
    
    # 5.6 Keep only letters, numbers, spaces, periods, and commas
    text = re.sub(r'[^\w\s\.\,]', ' ', text)
    
    # 5.7 Collapse multiple whitespace characters into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 5.8 Split into sentences and drop very short fragments (<20 chars)
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 20]
    
    # 5.9 Rejoin sentences with a period and a space
    return '. '.join(sentences)

# 6. Apply the embedding-cleaning function to the merged text
merged_df['Text_For_Embeddings'] = merged_df['OCR_Text_Merged'].apply(clean_for_embedding)

# 7. (Optional) Inspect the result
#print(merged_df[['group_id', 'OCR_Text_Merged', 'Text_For_Embeddings', 'Length_After']].head())
#merged_df.drop('OCR_Text_Merged', axis=1, inplace=True)

# 8. (Optional) Save the final DataFrame to Parquet for downstream use
merged_df.head(1)


Label                          group_id Format            Size  \
0      0  00163fe7688e71ce06f495a6811fef71   JPEG  1224 x 1584 px   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [34]:
import os
import re
import pandas as pd

# Ruta base donde se crearán los parquet
BASE_PATH = r"C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode"

# Lista de umbrales que quieras probar
WORD_THRESHOLD_LIST = [50, 150, 300, 450]  # puedes cambiar o extender esta lista

def split_into_sections(text: str):
    pattern = r'(?m)^(?P<title>[A-Z][A-Za-z0-9\s&\-]+)\n'
    splits = list(re.finditer(pattern, text))
    if not splits:
        return [("FullText", text)]
    sections = []
    for idx, match in enumerate(splits):
        title = match.group('title').strip()
        start = match.end()
        end = splits[idx+1].start() if idx+1 < len(splits) else len(text)
        body = text[start:end].strip()
        sections.append((title, body))
    return sections

def words(text: str):
    return text.split()

def make_subchunks(word_list, size, overlap):
    subchunks = []
    start = 0
    length = len(word_list)
    while start < length:
        end = min(start + size, length)
        subchunks.append(word_list[start:end])
        if end == length:
            break
        start += size - overlap
    return subchunks

def hybrid_chunking(text: str, group_id: str, WORD_THRESHOLD: int, OVERLAP: int):
    output = []
    for sec_title, sec_body in split_into_sections(text):
        wlist = words(sec_body)
        if len(wlist) <= WORD_THRESHOLD:
            output.append({
                'group_id': group_id,
                'section': sec_title,
                'chunk_id': f"{sec_title}_1",
                'chunk_text': " ".join(wlist)
            })
        else:
            subchs = make_subchunks(wlist, size=WORD_THRESHOLD, overlap=OVERLAP)
            for i, sub in enumerate(subchs, start=1):
                output.append({
                    'group_id': group_id,
                    'section': sec_title,
                    'chunk_id': f"{sec_title}_{i}",
                    'chunk_text': " ".join(sub)
                })
    return output

# Carga tu merged_df (ya debe tener ['group_id','Text_For_Embeddings'])
# from tu_origen import merged_df

# Ejemplo: merged_df = pd.read_parquet(...)

for threshold in WORD_THRESHOLD_LIST:
    chunk_size = threshold
    overlap = int(threshold * 0.15)
    
    all_chunks = []
    for _, row in merged_df.iterrows():
        chunks = hybrid_chunking(
            row['Text_For_Embeddings'],
            row['group_id'],
            WORD_THRESHOLD=threshold,
            OVERLAP=overlap
        )
        all_chunks.extend(chunks)
    
    df_chunks = pd.DataFrame(all_chunks)
    
    # Crea carpeta de salida si no existe
    out_dir = os.path.join(BASE_PATH, f"chunks_{threshold}")
    os.makedirs(out_dir, exist_ok=True)
    
    # Guarda en Parquet
    out_path = os.path.join(out_dir, f"wiki_hybrid_chunks_{threshold}.parquet")
    df_chunks.to_parquet(out_path, index=False)
    
    print(f"✅ Guardado chunks con umbral={threshold} en: {out_path}")


✅ Guardado chunks con umbral=50 en: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\chunks_50\wiki_hybrid_chunks_50.parquet
✅ Guardado chunks con umbral=150 en: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\chunks_150\wiki_hybrid_chunks_150.parquet
✅ Guardado chunks con umbral=300 en: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\chunks_300\wiki_hybrid_chunks_300.parquet
✅ Guardado chunks con umbral=450 en: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\chunks_450\wiki_hybrid_chunks_450.parquet
